# Silver: Limpeza, Tipagem e Enriquecimento dos Dados do Bitcoin

Este notebook corresponde à camada Silver da arquitetura Medallion, sendo responsável pela transformação e enriquecimento dos dados provenientes da camada Bronze.

Nesta etapa, os dados passam pelas seguintes transformações:

1. Conversão do timestamp em milissegundos para data legível (yyyy-MM-dd)
2. Renomeação das colunas para nomes semânticos (preco_usd, market_cap_usd, 
   volume_usd)
3. Remoção de duplicatas. A API CoinGecko pode retornar dois registros para o dia atual: um às 00h e outro no momento da consulta

Os dados transformados são persistidos no formato Delta Lake no container silver do ADLS Gen2, com carga incremental via MERGE utilizando a data como chave, garantindo que apenas registros novos sejam inseridos a cada execução diária.

In [0]:
%pip install azure-storage-blob

In [0]:
dbutils.library.restartPython()

In [0]:
dbutils.widgets.text("env", "prod")
dbutils.widgets.text("execution_date", "")

env            = dbutils.widgets.get("env")
execution_date = dbutils.widgets.get("execution_date")

print(f"Ambiente: {env}")
print(f"Data execução: {execution_date}")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

storage_account = dbutils.secrets.get(scope='kv-scope-bitcoin', key='adls-storage-account-name')
storage_key     = dbutils.secrets.get(scope='kv-scope-bitcoin', key='adls-storage-key-bitcoin')

BRONZE_PATH = f'abfss://bronze@{storage_account}.dfs.core.windows.net/bitcoin/'
SILVER_PATH = f'abfss://silver@{storage_account}.dfs.core.windows.net/bitcoin/'

print('Configuração concluída.')

In [0]:

try:

    #  Leitura da Bronze 
    df_bronze = spark.read.format('delta') \
        .option(f'fs.azure.account.key.{storage_account}.dfs.core.windows.net', storage_key) \
        .load(BRONZE_PATH)

    print(f'Registros na Bronze: {df_bronze.count()}')

    #Transformações Silver 

    # 1. Converte timestamp para data
    df_silver = df_bronze.withColumn(
        'data',
        F.to_date(F.from_unixtime(F.col('timestamp_ms') / 1000))
    )

    # 2. Renomeia colunas para nomes semânticos
    df_silver = df_silver \
        .withColumnRenamed('preco',      'preco_usd') \
        .withColumnRenamed('market_cap', 'market_cap_usd') \
        .withColumnRenamed('volume',     'volume_usd')

    # 3. Remove duplicatas mantendo o registro mais recente por data
    window_dedup = Window.partitionBy('data').orderBy(F.col('timestamp_ms').desc())
    df_silver = df_silver \
        .withColumn('rank', F.row_number().over(window_dedup)) \
        .filter(F.col('rank') == 1) \
        .drop('rank')

    # 4. Seleciona e ordena colunas finais
    df_silver = df_silver.select(
        'data',
        'timestamp_ms',
        'ativo',
        'moeda',
        'preco_usd',
        'market_cap_usd',
        'volume_usd',
        'data_carga_lake',
        'fonte'
    ).orderBy('data')

    total_silver = df_silver.count()
    print(f'Registros após transformação: {total_silver}')
    df_silver.limit(5).display()

    # Grava: overwrite na primeira carga, MERGE nas seguintes
    try:
        DeltaTable.forPath(spark, SILVER_PATH)
        is_first_load = False
    except:
        is_first_load = True

    if is_first_load:
        df_silver.write.format('delta') \
            .mode('overwrite') \
            .option('mergeSchema', 'true') \
            .option(f'fs.azure.account.key.{storage_account}.dfs.core.windows.net', storage_key) \
            .save(SILVER_PATH)
        print('Primeira carga: overwrite concluído.')

    else:
        delta_table = DeltaTable.forPath(spark, SILVER_PATH)

        delta_table.alias('target').merge(
            df_silver.alias('source'),
            'target.data = source.data'
        ) \
        .whenMatchedUpdateAll() \
        .whenNotMatchedInsertAll() \
        .execute()

        print('Carga incremental: MERGE concluído.')

    # Registra tabela no catálogo
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS silver.tb_bitcoin
        USING DELTA
        LOCATION '{SILVER_PATH}'
    """)

    total_final = spark.read.format('delta') \
        .option(f'fs.azure.account.key.{storage_account}.dfs.core.windows.net', storage_key) \
        .load(SILVER_PATH).count()

    print(f'Total na Silver após carga: {total_final}')
    dbutils.notebook.exit(f'SUCCESS: Silver concluída. Total: {total_final}')

except Exception as e:
    error_msg = str(e)
    if 'SUCCESS' in error_msg:
        dbutils.notebook.exit(error_msg)
    print(f'ERRO no Silver: {error_msg}')
    dbutils.notebook.exit(f'ERROR: Silver falhou. {error_msg}')